In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 37. Week 25 — Stationarity, AR diagnostics, and forecast evaluation

## 学習目標

- weak stationarityとergodicityを区別できる
- levelとdifferenceのACF/PACF、Dickey–Fuller diagnosticを比較できる
- AR forecastをrandom-walk baselineと同じvalidation originで評価できる
- ordinary t critical valueをDF statisticへ使わない理由を説明できる

## 前提知識

- lag operator、least squares、autocorrelation
- B5のchronological validation

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 37


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Stationarity contract

弱定常性は (E[y_t]=mu)、(operatorname{Cov}(y_t,y_{t-h})=gamma(h)) が時点に依存しないこと。ergodicityは一つの長いpathの時間平均が母集団量へ収束するための別条件である。

Dickey–Fuller回帰

$$
\Delta y_t=c+\gamma y_{t-1}+\varepsilon_t
$$

のt statisticは通常のStudent-t分布に従わない。本章APIは診断量だけを返し、未実装のcritical valueやp-valueを捏造しない。

In [4]:
train_ten_year = curve_yields[train_mask, 3]
train_change = np.diff(train_ten_year) * 100.0
lag_limit = 20
diagnostic_table = pd.DataFrame(
    {
        "lag": np.arange(lag_limit + 1),
        "level_acf": qt.autocorrelation(train_ten_year, lag_limit),
        "change_acf": qt.autocorrelation(train_change, lag_limit),
        "level_pacf": qt.partial_autocorrelation(train_ten_year, lag_limit),
        "change_pacf": qt.partial_autocorrelation(train_change, lag_limit),
    }
)
display(diagnostic_table.head(8))
display(
    pd.DataFrame(
        [
            {"series": "10y level", **qt.dickey_fuller_diagnostic(train_ten_year).__dict__},
            {"series": "10y change", **qt.dickey_fuller_diagnostic(train_change).__dict__},
        ]
    )
)

fig = go.Figure()
fig.add_bar(x=diagnostic_table["lag"], y=diagnostic_table["level_acf"], name="level")
fig.add_bar(x=diagnostic_table["lag"], y=diagnostic_table["change_acf"], name="change")
fig.update_layout(
    title="10y Treasury ACF: level versus publication-to-publication change",
    xaxis_title="Lag (publication observations)",
    yaxis_title="Sample autocorrelation",
    barmode="group",
    template="plotly_white",
)
fig.show()

,lag,level_acf,change_acf,level_pacf,change_pacf
0,0,1.000000,1.000000,1.000000,1.000000
1,1,0.997405,-0.024358,0.997405,-0.024358
2,2,0.994946,-0.015254,0.024874,-0.015857
3,3,0.992543,-0.024955,0.010410,-0.025740
4,4,0.990240,0.002230,0.018486,0.000725
5,5,0.987842,0.025534,-0.018187,0.024841
6,6,0.985301,0.036833,-0.029475,0.037584
7,7,0.982600,-0.065202,-0.033527,-0.062645


,series,coefficient,standard_error,t_statistic,n_observations,includes_intercept
0,10y level,-0.002040,0.00167,-1.221653,1654,True
1,10y change,-1.024359,0.02458,-41.673705,1653,True


## 2. Five-publication validation forecast

AR(1) parameterはtrainingで一度だけfitする。各validation originでは観察済みhistoryを更新するが、係数をvalidation outcomeへ合わせて再推定しない。

In [5]:
horizon = 5
ar_level = qt.fit_ar(train_ten_year, 1)
validation_origins = np.flatnonzero(
    (curve_dates > train_end_date)
    & (curve_dates <= validation_end_date)
    & (np.arange(curve_dates.size) + horizon < curve_dates.size)
)
validation_origins = validation_origins[curve_dates[validation_origins + horizon] <= validation_end_date]
actual = curve_yields[validation_origins + horizon, 3]
ar_prediction = np.array(
    [qt.forecast_ar(ar_level, curve_yields[: origin + 1, 3], horizon)[-1] for origin in validation_origins]
)
random_walk = curve_yields[validation_origins, 3]
forecast_table = pd.DataFrame(
    [
        {"model": "random walk", "rmse_bp": 100.0 * np.sqrt(np.mean((actual - random_walk) ** 2))},
        {"model": "AR(1) level", "rmse_bp": 100.0 * np.sqrt(np.mean((actual - ar_prediction) ** 2))},
    ]
)
display(forecast_table)

,model,rmse_bp
0,random walk,15.369856
1,AR(1) level,15.652197


## 3. 失敗モード

- levelとdifferenceを同じestimandとして比べる
- ACFのconfidence bandを次数選択の唯一の規則にする
- DF statisticへ通常のt critical valueを使う
- validationの各originでorderを選び直す
- RMSE差を経済的価値と呼ぶ

## 4. 段階別演習

### 基礎

1. AR(1)のstationarity条件を導出せよ。
2. level/changeのACF差を記述せよ。

### 標準

3. AR(1)とAR(2)をtraining/validationだけで比較せよ。
4. horizon 1と20でrandom-walkとの差を測れ。

### 研究

5. rolling-origin loss差へHAC standard errorを付ける設計を書け。

## 5. Exit Criteria

- [ ] stationarityとergodicityを区別した
- [ ] DF diagnosticを通常のt testと呼ばない
- [ ] publication horizonを使った
- [ ] random walkをbaselineに残した
- [ ] validation outcomeをorder選択以外へ漏らしていない

## 6. 出典


- [Forecasting: Principles and Practice — Stationarity and differencing](https://otexts.com/fpp3/stationarity.html)
- [Forecasting: Principles and Practice — ARIMA models](https://otexts.com/fpp3/arima.html)
- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)